## Семинар 7
# Тема: Отбор наилучшей модели из нескольких с наилучшими гиперпараметрами поиском по сетке

---

### Цель работы

Научиться выбирать наилучшую модель классификации из нескольких кандидатов с одновременным подбором гиперпараметров с помощью `GridSearchCV` и `Pipeline`.

## Теоретическая справка

### Проблема выбора модели

На практике заранее неизвестно, какой алгоритм подойдёт для конкретной задачи лучше всего. Поэтому применяют **одновременный перебор** нескольких моделей и их гиперпараметров. Это позволяет избежать ручного последовательного сравнения и получить статистически обоснованный выбор.

Ключевая идея: включить **сам класс модели** в пространство поиска наряду с её гиперпараметрами.

---

### Pipeline

[`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) — объект scikit-learn, объединяющий несколько шагов обработки данных и финальную модель в единую цепочку:

```
Данные → Шаг 1 (Scaler) → Шаг 2 (Encoder) → ... → Финальная модель
```

**Преимущества Pipeline:**

| Преимущество | Пояснение |
|---|---|
| Защита от утечки данных | Scaler обучается только на `X_train`, не видит `X_test` |
| Единый API | `fit`, `predict`, `score` работают как у обычной модели |
| Совместимость с GridSearchCV | Гиперпараметры любого шага доступны через `шаг__параметр` |
| Воспроизводимость | Весь процесс инкапсулирован в одном объекте |

**Синтаксис создания Pipeline:**

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),   # шаг 1: масштабирование
    ('classifier', SVC())           # шаг 2: классификатор
])
```

Доступ к гиперпараметрам шага осуществляется через двойное подчёркивание: `шаг__параметр`. Например: `classifier__C`, `scaler__with_mean`.

---

### GridSearchCV

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) выполняет исчерпывающий перебор всех комбинаций указанных гиперпараметров с оценкой качества через кросс-валидацию.

**Схема работы:**

```
Для каждой комбинации гиперпараметров:
    Разделить обучающие данные на cv фолдов
    Для каждого фолда:
        Обучить модель на (cv-1) фолдах
        Оценить на оставшемся фолде
    Усреднить оценку по всем фолдам
Выбрать комбинацию с наилучшей средней оценкой
Переобучить на всей обучающей выборке с лучшими параметрами
```

**Ключевые параметры GridSearchCV:**

| Параметр | Описание | Рекомендуемые значения |
|---|---|---|
| `estimator` | Модель или Pipeline | Любой объект sklearn |
| `param_grid` | Словарь или список словарей гиперпараметров | dict или list[dict] |
| `cv` | Число фолдов кросс-валидации | 5 (по умолчанию) |
| `scoring` | Метрика оценки | `'accuracy'`, `'f1'`, `'roc_auc'`, ... |
| `refit` | Переобучить лучшую модель на всех данных | `True` (по умолчанию) |
| `n_jobs` | Число параллельных задач | `-1` (все ядра) |
| `return_train_score` | Сохранять ли train score | `True` для диагностики |

**Ключевые атрибуты после обучения:**

| Атрибут | Описание |
|---|---|
| `best_params_` | Словарь лучших гиперпараметров |
| `best_score_` | Лучшее среднее CV-качество |
| `best_estimator_` | Переобученная лучшая модель |
| `cv_results_` | Подробная таблица всех экспериментов |

---

### Выбор модели через Pipeline + GridSearchCV

Чтобы перебирать **разные классы моделей**, шаг классификатора задаётся как гиперпараметр Pipeline. Тогда `param_grid` становится **списком словарей**, каждый из которых соответствует одной модели:

```python
pipe = Pipeline([('classifier', SVC())])  # placeholder

param_grid = [
    {'classifier': [SVC()],
     'classifier__C': [0.1, 1, 10]},

    {'classifier': [DecisionTreeClassifier()],
     'classifier__max_depth': [3, 5, 10]}
]

grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)
```

⚠️ **Важно:** GridSearchCV итерирует по словарям в `param_grid` независимо, поэтому параметры одной модели не смешиваются с параметрами другой.

---

### Добавление предобработки в Pipeline

Масштабирование нужно **не всем** моделям одинаково. Решения:

**Вариант 1.** Единый Scaler для всего Pipeline (применяется ко всем моделям):
```python
pipe = Pipeline([('scaler', MinMaxScaler()), ('classifier', SVC())])
```

**Вариант 2.** Вложенный Pipeline для конкретной модели:
```python
param_svc = {'classifier': [make_pipeline(MinMaxScaler(), SVC())],
             'classifier__svc__C': [0.1, 1, 10]}
param_tree = {'classifier': [DecisionTreeClassifier()],
              'classifier__max_depth': [3, 5, 10]}
```

Вариант 2 предпочтителен, когда одна модель требует масштабирования (SVM, KNN), а другая — нет (деревья, случайный лес).

---

### Дополнительные ресурсы

- 📖 [Pipeline — документация sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- 📖 [GridSearchCV — документация sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- 📖 [User Guide: Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- 📖 [User Guide: Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)

---

Рассмотрим следующую задачу: необходимо подобрать наилучшую модель (из нескольких) с наилучшими гиперпараметрами, а затем для выбранной модели оценить качество обобщающей способности. Это можно сделать с помощью поиска по сетке, в которой присутствует и вид модели тоже (а не только значения гиперпараметров модели) с перекрёстной проверкой.

Импортируем необходимые библиотеки:

In [25]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import GridSearchCV

### 1. Загрузите встроенный датасет load_breast_cancer. Обозначьте признаки за X, а целевую переменную за y и выведите их размеры. Выведите данные в виде датафрейма.

In [26]:
cancer= load_breast_cancer()
X=cancer.data
y=cancer.target
X.shape,y.shape

((569, 30), (569,))

In [27]:
df=pd.DataFrame(X,columns=cancer.feature_names)
df['target']=y
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [28]:
df['target'].value_counts()

target
1    357
0    212
Name: count, dtype: int64

### 2. Разбейте данные на два набора обучающий и тестовый при помощи train_test_split, зафиксировав random_state=0. Выведите размеры полученных наборов.

In [29]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=0)
X_train.shape[0],X_test.shape[0]

(426, 143)

### 3. Используя GridSearchCV, подберите наилучшую модель классификатора на обучающем наборе данных, выбирая между методом опорных векторов и деревом решений, подобрав наилучшие гиперпараметры для этих моделей, используя следующие сетки гиперпараметров. Выведите получившуюся модель с наилучшими значениями гиперпараметров.

In [30]:
param_svc= {'classifier': [SVC()], 
            'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]}
param_tree = {'classifier': [DecisionTreeClassifier(random_state = 42)],
              "classifier__max_depth": [1, 3, 5, 7, 10],
              "classifier__min_samples_leaf": [1, 3, 5, 10],
              "classifier__criterion": ["gini", "entropy"]}

In [31]:
param_grid=[param_svc,param_tree]

In [32]:
pipeline=Pipeline([('classifier',SVC())])

In [33]:
gs=GridSearchCV(pipeline,param_grid,cv=5,n_jobs=-1)

In [34]:
gs.fit(X_train,y_train)

,estimator,"Pipeline(step...ier', SVC())])"
,param_grid,"[{'classifier': [SVC()], 'classifier__C': [0.001, 0.01, ...]}, {'classifier': [DecisionTreeC...ndom_state=42)], 'classifier__criterion': ['gini', 'entropy'], 'classifier__max_depth': [1, 3, ...], 'classifier__min_samples_leaf': [1, 3, ...]}]"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [35]:
df=pd.DataFrame(gs.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_classifier__C,param_classifier__criterion,param_classifier__max_depth,param_classifier__min_samples_leaf,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003912,0.001469,0.002198,0.000820,SVC(),0.001,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.001}",0.627907,0.635294,0.623529,0.623529,0.623529,0.626758,0.004593,46
1,0.003622,0.001249,0.002425,0.001002,SVC(),0.010,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.01}",0.627907,0.670588,0.658824,0.682353,0.670588,0.662052,0.018623,45
2,0.002600,0.000890,0.001518,0.000604,SVC(),0.100,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.1}",0.895349,0.858824,0.882353,0.870588,0.894118,0.880246,0.013980,44
3,0.002958,0.000313,0.001712,0.000103,SVC(),1.000,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 1}",0.918605,0.870588,0.905882,0.882353,0.917647,0.899015,0.019307,35
4,0.002221,0.000584,0.001287,0.000414,SVC(),10.000,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 10}",0.965116,0.882353,0.905882,0.917647,0.905882,0.915376,0.027386,22
5,0.002506,0.000673,0.001164,0.000346,SVC(),100.000,NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 100}",0.976744,0.894118,0.941176,0.905882,0.929412,0.929466,0.028907,2
6,0.001636,0.000595,0.000441,0.000234,DecisionTreeClassifier(random_state=42),NaN,gini,1.0,1.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,36
7,0.001415,0.000419,0.000307,0.000123,DecisionTreeClassifier(random_state=42),NaN,gini,1.0,3.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,36
8,0.001794,0.000567,0.000462,0.000248,DecisionTreeClassifier(random_state=42),NaN,gini,1.0,5.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,36
9,0.002054,0.000461,0.000532,0.000191,DecisionTreeClassifier(random_state=42),NaN,gini,1.0,10.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,36


In [36]:
df[['params','mean_test_score']]

,params,mean_test_score
0,"{'classifier': SVC(), 'classifier__C': 0.001}",0.626758
1,"{'classifier': SVC(), 'classifier__C': 0.01}",0.662052
2,"{'classifier': SVC(), 'classifier__C': 0.1}",0.880246
3,"{'classifier': SVC(), 'classifier__C': 1}",0.899015
4,"{'classifier': SVC(), 'classifier__C': 10}",0.915376
5,"{'classifier': SVC(), 'classifier__C': 100}",0.929466
6,{'classifier': DecisionTreeClassifier(random_s...,0.887332
7,{'classifier': DecisionTreeClassifier(random_s...,0.887332
8,{'classifier': DecisionTreeClassifier(random_s...,0.887332
9,{'classifier': DecisionTreeClassifier(random_s...,0.887332


In [37]:
gs.best_params_

{'classifier': DecisionTreeClassifier(random_state=42),
 'classifier__criterion': 'gini',
 'classifier__max_depth': 5,
 'classifier__min_samples_leaf': 3}

### 4. Оцените качество наилучшей модели, выведя значения метрик accuracy и f1-score на тестовых данных.

In [38]:
y_pred=gs.predict(X_test)
accuracy=metrics.accuracy_score(y_test,y_pred)
accuracy

0.9300699300699301

### 5. Добавьте масштабирование данных методом min-max в метод опорных векторов и снова подберите наилучшую модель классификатора, выбирая между методом опорных векторов и деревом решений с теми же диапазонами гиперпараметров. Выведите получившуюся модель с наилучшими значениями гиперпараметров.

In [39]:
param_svc= {'classifier': [SVC()], 
            'scaler':[MinMaxScaler()],
            'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]}
param_tree = {'classifier': [DecisionTreeClassifier(random_state = 42)],
              'scaler':[None],
              "classifier__max_depth": [1, 3, 5, 7, 10],
              "classifier__min_samples_leaf": [1, 3, 5, 10],
              "classifier__criterion": ["gini", "entropy"]}

In [40]:
param_grid=[param_svc,param_tree]

In [41]:
pipe=Pipeline([('scaler',MinMaxScaler()),('classifier',SVC())])

In [42]:
gs_new=GridSearchCV(pipe,param_grid,cv=5,n_jobs=-1)
gs_new.fit(X_train,y_train)

df_new=pd.DataFrame(gs_new.cv_results_)
df_new

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_classifier__C,param_scaler,param_classifier__criterion,param_classifier__max_depth,param_classifier__min_samples_leaf,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002737,0.000187,0.001387,0.000135,SVC(),0.001,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.001, ...",0.627907,0.635294,0.623529,0.623529,0.623529,0.626758,0.004593,45
1,0.002365,0.000189,0.001286,0.000078,SVC(),0.010,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.01, '...",0.627907,0.635294,0.623529,0.623529,0.623529,0.626758,0.004593,45
2,0.001566,0.000089,0.000909,0.000182,SVC(),0.100,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 0.1, 's...",0.895349,0.964706,0.941176,0.952941,0.976471,0.946129,0.027983,4
3,0.001356,0.000208,0.000583,0.000060,SVC(),1.000,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 1, 'sca...",0.976744,0.976471,0.952941,1.000000,1.000000,0.981231,0.017594,1
4,0.001224,0.000142,0.000463,0.000021,SVC(),10.000,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 10, 'sc...",1.000000,0.964706,0.952941,0.988235,0.988235,0.978824,0.017291,2
5,0.001080,0.000069,0.000428,0.000063,SVC(),100.000,MinMaxScaler(),NaN,NaN,NaN,"{'classifier': SVC(), 'classifier__C': 100, 's...",0.976744,0.941176,0.929412,0.976471,0.941176,0.952996,0.019752,3
6,0.001418,0.000278,0.000332,0.000112,DecisionTreeClassifier(random_state=42),NaN,None,gini,1.0,1.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,37
7,0.001251,0.000133,0.000321,0.000131,DecisionTreeClassifier(random_state=42),NaN,None,gini,1.0,3.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,37
8,0.001100,0.000080,0.000218,0.000022,DecisionTreeClassifier(random_state=42),NaN,None,gini,1.0,5.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,37
9,0.001337,0.000190,0.000271,0.000047,DecisionTreeClassifier(random_state=42),NaN,None,gini,1.0,10.0,{'classifier': DecisionTreeClassifier(random_s...,0.883721,0.894118,0.894118,0.905882,0.858824,0.887332,0.015887,37


In [44]:
gs_new.best_params_

{'classifier': SVC(), 'classifier__C': 1, 'scaler': MinMaxScaler()}

In [45]:
best_estimator=gs_new.best_estimator_
best_estimator

,steps,"[('scaler', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,feature_range,"(0, ...)"
,copy,True
,clip,False
,C,1
,kernel,'rbf'
,degree,3
,gamma,'scale'


### 6. Оцените качество полученной наилучшей модели с учётом масштабирования данных, выведя значения метрик accuracy и f1-score на тестовых данных. Сделайте вывод о том нужно ли было делать масштабирование данных.

In [46]:
y_pred=gs_new.predict(X_test)
accuracy=metrics.accuracy_score(y_test,y_pred)
f1=metrics.f1_score(y_test,y_pred)
accuracy,f1

(0.972027972027972, 0.978021978021978)

## Задания для самостоятельного выполнения
1. Загрузите встроенный датасет `load_iris`. Обозначьте данные за `X`, а целевую переменную за `y`.
2. Разбейте данные на два набора: обучающий и тестовый, взяв в тестовый набор 20% данных и указав random_state=0.
3. Используя `GridSearchCV`, подберите наилучшую модель классификатора на обучающем наборе данных, выбирая между методом логистической регрессии и деревом со следующими наборами гиперпараметров: param_grid = [{'classifier': [LogisticRegression(max_iter=500)],'classifier__C': np.logspace(0,4,10), 'classifier__penalty': ['l2', 'l1'], 'classifier__solver': ['liblinear']}, {'classifier': [LogisticRegression(max_iter=500)],'classifier__C': np.logspace(0,4,10), 'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs', 'newton-cg']}, {'classifier': [DecisionTreeClassifier(random_state = 42)], 'classifier__max_depth': [1, 3, 5, 7], 'classifier__criterion': ["gini", "entropy"], 'classifier__max_features': [1, 2, 3]}]. Выведите получившуюся модель с наилучшими значениями гиперпараметров.
4. Оцените качество наилучшей модели, выведя значения метрик accuracy и f1-score на тестовых данных.
5. Добавьте стандартную нормализацию данных в метод логистической регрессии и снова подберите наилучшую модель классификатора, выбирая между методом логистической регрессии и деревом решений с теми же диапазонами гиперпараметров. Выведите получившуюся модель с наилучшими значениями гиперпараметров.
6. Оцените качество полученной наилучшей модели с учётом стандартной нормализации данных, выведя значения метрик accuracy и f1-score на тестовых данных. Сделайте вывод о том нужна ли здесь была нормализация данных.